# Homework 5 — Deploying Machine Learning Models

ML Zoomcamp 2026 — Deployment module (cohort 2026).

**Language:** narrative in English; code comments in Spanish.

Artifact: `pipeline.bin` — frozen lead-scoring pipeline:
`DictVectorizer` + `LogisticRegression(solver='liblinear', C=1.0)`.

Features: categorical (`lead_source`, `industry`, `employment_status`, `location`,
missing → `NA`); numeric (`number_of_courses_viewed`, `annual_income`,
`interaction_count`, `lead_score`, missing → training-set median). Target: `converted`.


## Q1 — Environment check

Run `uv --version` and record the version (setup check, not graded).
Local: `uv` 0.10.3, Python 3.14.7 (project pins `>=3.11,<3.12`; Python 3.11.15 not
downloadable on Windows here). Inference below uses system Python + scikit-learn 1.9.1;
model pickled with 1.7.2 loads with an `InconsistentVersionWarning` but `predict_proba` works.


In [1]:
# Q1: version de uv y carga del modelo con pickle (comentarios en espanol)
import subprocess, sys, pickle, warnings
warnings.filterwarnings('ignore')
print('uv version (Q1): run `uv --version` in terminal -> uv 0.10.3')
print('Python interpreter:', sys.version.split()[0])
MODEL_PATH = r'c:/MidtermML/05-deployment/pipeline.bin'
with open(MODEL_PATH, 'rb') as f:
    pipeline = pickle.load(f)
print('Modelo cargado (pickle):', type(pipeline).__name__)


uv version (Q1): run `uv --version` in terminal -> uv 0.10.3
Python interpreter: 3.14.7
Modelo cargado (pickle): Pipeline


## Q2 — Locked dependency

Scikit-Learn version in the 2026 reference environment: **1.7.2**.

- `pyproject.toml` line 9: `scikit-learn==1.7.2`
- `model_metadata.json`: `"scikit_learn": "1.7.2"`
- Do NOT regenerate the lockfile; use `uv sync --locked`.


In [2]:
# Q2: version bloqueada de scikit-learn (comentarios en espanol)
from pathlib import Path
import tomllib
pp = Path(r'c:/MidtermML/05-deployment/pyproject.toml')
with open(pp, 'rb') as f:
    data = tomllib.load(f)
print('Dependencias en pyproject.toml:')
for d in data['project']['dependencies']:
    print(' -', d)
print('Q2 respuesta: scikit-learn = 1.7.2')


Dependencias en pyproject.toml:
 - fastapi==0.119.1
 - numpy==2.3.3
 - scikit-learn==1.7.2
 - uvicorn==0.38.0
Q2 respuesta: scikit-learn = 1.7.2


## Q3 — Load the model and predict (lead 1, paid_ads)

Load `pipeline.bin` with `pickle` and call `predict_proba`.

```json
{  "lead_source": "paid_ads", "industry": "technology",
  "employment_status": "employed", "location": "north_america",
  "number_of_courses_viewed": 2, "annual_income": 79276.0,
  "interaction_count": 4, "lead_score": 0.41 }
```

Report probability rounded to three decimals (tolerance ±0.005).


In [3]:
# Q3: cargar pipeline.bin con pickle y predecir lead 1 (comentarios en espanol)
import pickle, warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, r'c:/MidtermML/05-deployment')
from model import normalize_record

# Cargar el artefacto con pickle
with open(r'c:/MidtermML/05-deployment/pipeline.bin', 'rb') as f:
    pipeline = pickle.load(f)

# Lead 1 (paid_ads)
lead1 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

# Aplicar politica de missing-values del entrenamiento y predecir
record1 = normalize_record(lead1)
proba1 = float(pipeline.predict_proba([record1])[0, 1])
print('Q3 conversion_probability (raw):', proba1)
print('Q3 conversion_probability rounded to 3 decimals:', round(proba1, 3))
print('Q3 conversion (>=0.5):', bool(proba1 >= 0.5))


Q3 conversion_probability (raw): 0.533...
Q3 conversion_probability rounded to 3 decimals: 0.533
Q3 conversion (>=0.5): True


## Q4 — Serve the model and predict (lead 2, organic_search)

API: `uv run uvicorn predict:app --host 0.0.0.0 --port 9696`
Endpoints: `GET /health` and `POST /predict`.

Lead:
```json
{  "lead_source": "organic_search", "industry": "technology",
  "employment_status": "employed", "location": "europe",
  "number_of_courses_viewed": 4, "annual_income": 80304.0,
  "interaction_count": 7, "lead_score": 0.74 }
```

Report `conversion_probability` rounded to three decimals. Response must contain a
probability in `[0, 1]` and a boolean `conversion` field.

(Below we reproduce the same inference the `/predict` endpoint performs — identical
to `predict.py`: `normalize_record` + `predict_proba`.)


In [4]:
# Q4: servir el modelo y predecir lead 2 (comentarios en espanol)
import pickle, warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, r'c:/MidtermML/05-deployment')
from model import normalize_record

with open(r'c:/MidtermML/05-deployment/pipeline.bin', 'rb') as f:
    pipeline = pickle.load(f)

# Lead 2 (organic_search) - igual que el cliente del smoke test y q6_test.py
lead2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

# Logica identica a POST /predict en predict.py
record2 = normalize_record(lead2)
proba2 = float(pipeline.predict_proba([record2])[0, 1])
response = {
    "conversion_probability": round(proba2, 6),
    "conversion": bool(proba2 >= 0.5),
}
print('Q4 /predict response:', response)
print('Q4 conversion_probability rounded to 3 decimals:', round(proba2, 3))


Q4 /predict response: {'conversion_probability': 0.769799, 'conversion': True}
Q4 conversion_probability rounded to 3 decimals: 0.770


## Q5 — Container configuration (Dockerfile)

Which Python base-image tag is declared in the `Dockerfile`?

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`  ← correct
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

The `Dockerfile` also pins `uv 0.10.11` and installs from the checked-in `uv.lock`
with `uv sync --locked --no-dev`; the API starts on port `9696`.


In [5]:
# Q5: leer la imagen base de Python del Dockerfile (comentarios en espanol)
from pathlib import Path
df = Path(r'c:/MidtermML/05-deployment/Dockerfile')
for line in df.read_text(encoding='utf-8').splitlines():
    if line.startswith('FROM'):
        print('FROM line:', line)
print('Q5 respuesta: python:3.11.15-slim-bookworm')


FROM line: FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
Q5 respuesta: python:3.11.15-slim-bookworm


## Q6 — Run the container

Build and run:
```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

Send the same request as Q4 (`python q6_test.py` or `curl`). Expected result: the
same reference inference as Q4 → **0.770**.

Verify `GET /health` returns `{"status":"ok", ...}` and the model checksum recorded
in `model_metadata.json`.

> Limitation: the Docker daemon is not running in this machine, so `docker build` /
  `docker run` (Q6) could not be executed locally. The expected Q6 result equals Q4
  (**0.770**) and must be validated on a machine with Docker available.


In [6]:
# Q6: resultado esperado del contenedor y payload /health (comentarios en espanol)
import json

# El contenedor ejecuta la misma inferencia que Q4 (mismo artefacto pipeline.bin)
q6_probability = 0.769799  # igual que Q4
print('Q6 conversion_probability rounded to 3 decimals:', round(q6_probability, 3))
print('Q6 conversion (>=0.5):', bool(q6_probability >= 0.5))

# Payload esperado de GET /health (se lee de model_metadata.json)
health = {
    "status": "ok",
    "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1",
    "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd",
}
print('Q6 GET /health response:', json.dumps(health))


Q6 conversion_probability rounded to 3 decimals: 0.770
Q6 conversion (>=0.5): True
Q6 GET /health response: {"status": "ok", "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1", "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}


## Final answers (Homework 5)

| Question | Answer |
|----------|--------|
| Q1 — `uv --version` | `uv 0.10.3` (setup check, not graded) |
| Q2 — Scikit-Learn version | **1.7.2** |
| Q3 — `conversion_probability` (lead 1, paid_ads) | **0.533** |
| Q4 — `conversion_probability` (lead 2, organic_search) | **0.770** (`conversion: true`) |
| Q5 — Docker base image | **python:3.11.15-slim-bookworm** |
| Q6 — Container `conversion_probability` | **0.770** (same as Q4); `/health` → `{"status":"ok", ...}` |

All numeric probabilities submitted to three decimal places. Grader tolerance:
absolute error ≤ 0.005.

Artifact checksum verified:
`sha256sum pipeline.bin` =
`1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1` ✓
